# Comparaison des voix wolof : Adia vs Oolel-Voices vs Kiriku-Wolof-TTS

Génère les **mêmes 15 phrases** avec deux autres modèles wolof, pour comparer à l'oreille avec Adia (la voix actuelle de GuindiMa).

**Avant de lancer :**
1. Exécution → Modifier le type d'exécution → **GPU T4**.
2. Secret Colab `HF_TOKEN` avec « Accès au notebook » activé.
3. Sur https://huggingface.co/AIHubSN/Kiriku-Wolof-TTS, connecte-toi et accepte les conditions d'accès (modèle protégé).
4. Importe dans le panneau Fichiers un audio d'Adia, par exemple `data/tts_cache/aucune_ligne.wav` : Oolel-Voices s'en sert pour **imiter la même voix**.

Puis **Exécution → Tout exécuter**. Si un modèle échoue, les autres continuent. À la fin, `comparaison_voix.zip` est téléchargé.

In [ ]:
PHRASES = [
 [
  "01_ucad_fr",
  "Université Cheikh Anta Diop"
 ],
 [
  "02_ucad_wo",
  "Iniwersite Seex Anta Joob"
 ],
 [
  "03_poste_thiaroye_fr",
  "Poste Thiaroye"
 ],
 [
  "04_liberte_six",
  "Liberté six"
 ],
 [
  "05_keur_massar",
  "Keur Massar"
 ],
 [
  "06_guediawaye",
  "Guédiawaye"
 ],
 [
  "07_parcelles",
  "Parcelles Assainies"
 ],
 [
  "08_rond_point_cambérène",
  "Rond-Point Cambérène"
 ],
 [
  "09_tata",
  "Jëlal bus bu Tata, numero fanweer ak juróom ñaar."
 ],
 [
  "10_taf_taf",
  "Jëlal bus bu Dem Dikk, TAF TAF."
 ],
 [
  "11_phrase_complete",
  "Jëlal bus bu Dem Dikk, numero ñaari téeméer ak ñaar fukk ak benn, nga wàcc Gadaye. Ci ginnaaw, jëlal bus bu Tata, numero fanweer ak juróom ñaar, moo lay yóbbu ba Malika."
 ],
 [
  "12_universite_phrase",
  "Jëlal bus bu Dem Dikk, numero benn, moo lay yóbbu ba Université Cheikh Anta Diop."
 ],
 [
  "13_aeroport",
  "Aéroport Léopold Sédar Senghor"
 ],
 [
  "14_aibd",
  "Aéroport AIBD de Diass"
 ],
 [
  "15_pont_emergence",
  "Pont de l'Émergence"
 ]
]
import os
os.makedirs('sortie/oolel', exist_ok=True)
os.makedirs('sortie/kiriku', exist_ok=True)
print(len(PHRASES), 'phrases')

In [ ]:
# Token Hugging Face (secret Colab)
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = (userdata.get('HF_TOKEN') or '').strip()
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('connecté')

## 1. Oolel-Voices (soynade-research) — avec la voix d'Adia comme modèle

In [ ]:
!pip -q install 'librosa>=0.10.2' transformers==4.46.3 diffusers==0.29.0 conformer==0.3.2 torchcodec s3tokenizer num2words

In [ ]:
import glob, torch, torchaudio, traceback
from huggingface_hub import snapshot_download
from transformers import AutoModel
try:
    ref = (glob.glob('/content/aucune_ligne.wav') + glob.glob('/content/*.wav'))[0]
    print('voix de référence :', ref)
    path = snapshot_download(repo_id='soynade-research/Oolel-Voices')
    oolel = AutoModel.from_pretrained(path, trust_remote_code=True)
    for nom, texte in PHRASES:
        wav = oolel.generate(texte, audio_prompt_path=ref, cfg_weight=0.5, exaggeration=0.2, temperature=0.3)
        torchaudio.save(f'sortie/oolel/{nom}.wav', wav.cpu() if hasattr(wav,'cpu') else wav, oolel.sr)
        print('ok', nom)
except Exception:
    traceback.print_exc()
    print('>>> Oolel-Voices a échoué : envoie cette erreur à Claude, la suite continue.')

## 2. Kiriku-Wolof-TTS (AI Hub Sénégal) — sa propre voix

In [ ]:
!pip -q install coqui-tts

In [ ]:
import traceback
try:
    from huggingface_hub import snapshot_download
    from TTS.utils.synthesizer import Synthesizer
    d = snapshot_download(repo_id='AIHubSN/Kiriku-Wolof-TTS', local_dir='kiriku', token=os.environ['HF_TOKEN'])
    synth = Synthesizer(tts_checkpoint='kiriku/model.pth', tts_config_path='kiriku/config.json', use_cuda=True)
    for nom, texte in PHRASES:
        wav = synth.tts(text=texte)
        synth.save_wav(wav, f'sortie/kiriku/{nom}.wav')
        print('ok', nom)
except Exception:
    traceback.print_exc()
    print('>>> Kiriku-Wolof-TTS a échoué : envoie cette erreur à Claude.')

## 3. Télécharger

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('comparaison_voix', 'zip', 'sortie')
files.download('comparaison_voix.zip')